# Tier 2 — T2.3: Inverse Meta-Heads Analysis (Paper 3)

Paper 3's main sweep produced **20 "inverse meta-heads"** — heads whose ablation *improves* self-consistency on disagreement questions (Δ_self < 0). They were observed but not discussed.

This notebook does three things:

1. **Characterize the 20 inverse heads** — Paper 2 class distribution, layer position, Δ_self/Δ_cross/Δ_task signature.
2. **Disagreement pattern analysis** — on which questions does ablation help?
3. **De-biasing hypothesis test** — do inverse-meta heads lock in specific answer patterns? Measure **answer variance across sampling seeds** with vs without ablation. Hypothesis: ablation decreases variance (removes a "de-biasing" component) OR increases it (removes a "locked" bias). Either direction is informative.

**Compute:** ~15 min A100. Loads existing Paper 3 data from GitHub, only the variance probe needs model forward passes.

**Output:** `tier2_t23_inverse_meta.csv` (variance probe results) + `tier2_t23_summary.json`.

In [ ]:
!pip install -q transformers datasets torch accelerate scipy pandas

In [ ]:
import torch, json, os, time, csv, hashlib, gc, urllib.request
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}  |  Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = '/content/drive/MyDrive/DFE_research/tier2'
    os.makedirs(OUT_DIR, exist_ok=True)
    test = os.path.join(OUT_DIR, '.w')
    with open(test, 'w') as f: f.write('ok')
    os.remove(test)
    print(f'Drive mounted and verified; output to {OUT_DIR}')
except Exception as e:
    raise RuntimeError(f'Drive mount required: {e}')

GITHUB_RAW = 'https://raw.githubusercontent.com/mool32/functional-differentiation-dfe/main'

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

## Load existing Paper 3 micropilot ablation data

Looks in three places in order:
1. User's Drive: `MyDrive/DFE_research/preflight/micropilot_ablations.csv` (produced by `micropilot_ablation_sweep.ipynb`)
2. GitHub raw (if eventually uploaded there)
3. Current Colab working directory (if user uploaded manually)

If nothing found, prints clear instructions and stops.

In [ ]:
abl_path = None
CANDIDATES_ABL = [
    '/content/drive/MyDrive/DFE_research/preflight/micropilot_ablations.csv',
    '/content/drive/MyDrive/DFE_research/tier2/micropilot_ablations.csv',
    os.path.join(OUT_DIR, 'micropilot_ablations.csv'),
    '/content/micropilot_ablations.csv',
]
for p in CANDIDATES_ABL:
    if os.path.exists(p):
        abl_path = p
        log(f'Found ablation CSV at {p}')
        break

if abl_path is None:
    # Last resort: try GitHub
    try:
        remote = f'{GITHUB_RAW}/data/micropilot/micropilot_ablations.csv'
        local = os.path.join(OUT_DIR, 'micropilot_ablations.csv')
        urllib.request.urlretrieve(remote, local)
        abl_path = local
        log(f'Downloaded from GitHub to {local}')
    except Exception as e:
        raise RuntimeError(
            'micropilot_ablations.csv not found. Expected locations:\n'
            + '\n'.join(f'  - {p}' for p in CANDIDATES_ABL)
            + '\n\nFix: copy the CSV from your previous Paper 3 run into one of these paths,'
            ' then re-run this cell. The file is produced by micropilot_ablation_sweep.ipynb'
            ' and normally lives at MyDrive/DFE_research/preflight/.'
        )

df = pd.read_csv(abl_path)
log(f'Loaded {len(df)} ablations')

# Inverse meta-heads: Δ_self < threshold (ablation improves self-alignment)
# Use same convention as Paper 3: delta_self = baseline_self - ablated_self
# so POSITIVE = ablation hurt self (classic meta). NEGATIVE = ablation helped self (inverse).
df_sorted = df.sort_values('delta_self').reset_index(drop=True)
inverse = df_sorted.head(20).copy()
print('\n=== 20 INVERSE META-HEADS (lowest Delta_self = ablation improves self-consistency) ===')
print(f'{"L":>3} {"H":>3} | {"Δself":>7} {"Δcross":>7} {"Δtask":>7}')
print('-' * 50)
for _, r in inverse.iterrows():
    print(f'{int(r["layer_idx"]):>3} {int(r["head_idx"]):>3} | '
          f'{r["delta_self"]:>+7.3f} {r["delta_cross"]:>+7.3f} {r["delta_task"]:>+7.4f}')

## Paper 2 class distribution of inverse heads

In [ ]:
p2_path = os.path.join(OUT_DIR, 'paper2_all_ablations.csv')
urllib.request.urlretrieve(f'{GITHUB_RAW}/data/all_ablations.csv', p2_path)
p2 = pd.read_csv(p2_path)
p2_heads = p2[p2['perturbation_type'] == 'head'].copy()

classes = {}
for (L, H), g in p2_heads.groupby(['layer_idx', 'head_idx']):
    g = g.sort_values('checkpoint')
    init = abs(g.iloc[0]['delta'])
    final = abs(g.iloc[-1]['delta'])
    if final < 5e-4:
        cls = 'never-critical'
    elif init > 5e-4 and final > 5e-4:
        cls = 'born-critical'
    elif init < 1e-4 and final > 5e-4:
        cls = 'emergent'
    else:
        cls = 'growing'
    classes[(int(L), int(H))] = cls

inverse['paper2_class'] = inverse.apply(
    lambda r: classes.get((int(r['layer_idx']), int(r['head_idx'])), 'unknown'), axis=1
)

# Class distribution + enrichment vs all 144 sampled heads
all_classes = pd.Series([classes.get((L, H), 'unknown')
                         for L in range(24) for H in [0,3,6,9,12,15]]).value_counts()
inv_dist = inverse['paper2_class'].value_counts()
print('\n=== PAPER 2 CLASS OF INVERSE META-HEADS ===')
print(f'{"class":<15} {"obs":>4} {"exp":>6} {"ratio":>6}')
print('-' * 40)
for cls in ['born-critical', 'emergent', 'growing', 'never-critical', 'unknown']:
    obs = inv_dist.get(cls, 0)
    exp = 20 * all_classes.get(cls, 0) / all_classes.sum()
    ratio = obs / exp if exp > 0 else float('inf')
    print(f'{cls:<15} {obs:>4} {exp:>6.1f} {ratio:>6.2f}x')

# Layer distribution
print('\n=== LAYER DISTRIBUTION ===')
layer_dist = inverse['layer_idx'].value_counts().sort_index()
for L, c in layer_dist.items():
    print(f'  L{int(L):>2}: {"#" * int(c)} ({int(c)})')

## De-biasing hypothesis — variance across sampling seeds

For each inverse-meta head, generate 10 completions per question with multinomial sampling at different seeds, with and without ablation. Measure variance (entropy) of first-token distribution across seeds.

Hypothesis A (de-biasing): ablation → answer variance *decreases* → head was a noise/exploration component.
Hypothesis B (bias-locking): ablation → answer variance *increases* → head was enforcing a preference.
Null: no change → head's Δ_self was a spurious small effect.

Probe design: 5 questions × 10 seeds × 20 heads × 2 (with/without abl) = 2,000 forward passes @ ~0.05s = ~2 min.

In [ ]:
MODEL_NAME = 'EleutherAI/pythia-410m-deduped'
log(f'Loading {MODEL_NAME} step143000 (float32)...')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision='step143000', torch_dtype=torch.float32
).to(device).eval()

# Load question bank
qb_path = os.path.join(OUT_DIR, 'question_bank.json')
urllib.request.urlretrieve(f'{GITHUB_RAW}/data/micropilot/question_bank.json', qb_path)
with open(qb_path) as f:
    qb = json.load(f)

# Subset: use 8 questions (enough for variance estimation, keeps cost low)
PROBE_QUESTIONS = qb['questions'][:8]
log(f'Probing {len(PROBE_QUESTIONS)} questions')

PHASE_A = ("If I asked you 'What is 5 + 3?', your first word would be: Eight\n"
           "If I asked you 'Who wrote Hamlet?', your first word would be: Shakespeare\n"
           "If I asked you 'What color is grass?', your first word would be: Green\n"
           "If I asked you '{q}', your first word would be:")

def ablate_head(L, H):
    hd = model.config.hidden_size // model.config.num_attention_heads
    w = model.gpt_neox.layers[L].attention.dense.weight
    saved = w.data.clone()
    w.data[:, H*hd:(H+1)*hd] = 0
    return saved

def restore_head(L, saved):
    model.gpt_neox.layers[L].attention.dense.weight.data.copy_(saved)

@torch.no_grad()
def first_token_distribution_entropy(prompt, n_samples=10, temperature=1.0):
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    logits = model(**inputs).logits[0, -1, :] / temperature
    # Sample n_samples first tokens, compute entropy of their empirical distribution
    probs = torch.softmax(logits, dim=-1)
    samples = torch.multinomial(probs, n_samples, replacement=True).cpu().numpy()
    unique, counts = np.unique(samples, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log(p + 1e-12)))  # empirical entropy

In [ ]:
CSV_PROBE = os.path.join(OUT_DIR, 'tier2_t23_inverse_meta.csv')
FIELDS = ['layer_idx', 'head_idx', 'q_idx', 'question', 'baseline_entropy', 'ablated_entropy', 'delta_entropy']

completed = set()
if os.path.exists(CSV_PROBE):
    _d = pd.read_csv(CSV_PROBE)
    completed = set(zip(_d['layer_idx'], _d['head_idx'], _d['q_idx']))
    log(f'{len(completed)} probe rows already done')

def append_probe(row):
    new = not os.path.exists(CSV_PROBE) or os.path.getsize(CSV_PROBE) == 0
    with open(CSV_PROBE, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new: w.writeheader()
        w.writerow(row)

for _, row in inverse.iterrows():
    L, H = int(row['layer_idx']), int(row['head_idx'])
    for qi, q in enumerate(PROBE_QUESTIONS):
        if (L, H, qi) in completed: continue
        prompt = PHASE_A.format(q=q['q'])
        torch.manual_seed(7000 + qi)
        ent_baseline = first_token_distribution_entropy(prompt, n_samples=10)
        saved = ablate_head(L, H)
        torch.manual_seed(7000 + qi)  # same multinomial seed for fair comparison
        ent_ablated = first_token_distribution_entropy(prompt, n_samples=10)
        restore_head(L, saved)
        append_probe({
            'layer_idx': L, 'head_idx': H, 'q_idx': qi, 'question': q['q'],
            'baseline_entropy': ent_baseline, 'ablated_entropy': ent_ablated,
            'delta_entropy': ent_ablated - ent_baseline,
        })
    log(f'  probed L{L}H{H}')

probe = pd.read_csv(CSV_PROBE)
log(f'probe done, {len(probe)} rows')

## Verdict on de-biasing hypothesis

In [ ]:
# Per-head mean Δ_entropy, aggregated across questions
per_head = probe.groupby(['layer_idx', 'head_idx'])['delta_entropy'].agg(['mean', 'std', 'count']).reset_index()
per_head['abs_mean'] = per_head['mean'].abs()

print('=== PER-HEAD ENTROPY CHANGE ===')
print(f'{"L":>3} {"H":>3} | {"mean Δentropy":>14} {"std":>7} | direction')
print('-' * 55)
for _, r in per_head.sort_values('mean').iterrows():
    direction = 'VARIANCE DOWN (de-biasing)' if r['mean'] < -0.05 else ('VARIANCE UP (bias-locked)' if r['mean'] > 0.05 else 'null')
    print(f'{int(r["layer_idx"]):>3} {int(r["head_idx"]):>3} | {r["mean"]:>+14.4f} {r["std"]:>+7.4f} | {direction}')

mean_delta = probe['delta_entropy'].mean()
n_de = (per_head['mean'] < -0.05).sum()
n_lock = (per_head['mean'] > 0.05).sum()
n_null = 20 - n_de - n_lock

# One-sample t-test on mean delta across all probes
t_stat, p_val = sp_stats.ttest_1samp(probe['delta_entropy'].values, 0)

print(f'\nGrand mean Δentropy: {mean_delta:+.4f}')
print(f'One-sample t-test H0=0: t={t_stat:.2f}, p={p_val:.4f}')
print(f'De-biasing heads: {n_de}/20   Bias-locking: {n_lock}/20   Null: {n_null}/20')

if n_de >= 12 and p_val < 0.05 and mean_delta < 0:
    verdict = 'DE-BIASING HYPOTHESIS SUPPORTED'
elif n_lock >= 12 and p_val < 0.05 and mean_delta > 0:
    verdict = 'BIAS-LOCKING HYPOTHESIS SUPPORTED'
else:
    verdict = 'NEITHER CLEAN — report as open question'

print(f'\n>>> T2.3 VERDICT: {verdict} <<<')

summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'n_inverse_heads': 20,
    'paper2_class_distribution': inv_dist.to_dict(),
    'layer_distribution': layer_dist.to_dict(),
    'grand_mean_delta_entropy': float(mean_delta),
    't_stat': float(t_stat), 'p_value': float(p_val),
    'n_de_biasing': int(n_de), 'n_bias_locking': int(n_lock), 'n_null': int(n_null),
    'verdict': verdict,
    'inverse_heads': [
        {'layer': int(r['layer_idx']), 'head': int(r['head_idx']),
         'delta_self': float(r['delta_self']), 'delta_cross': float(r['delta_cross']),
         'delta_task': float(r['delta_task']), 'paper2_class': r['paper2_class']}
        for _, r in inverse.iterrows()
    ],
}
summary_path = os.path.join(OUT_DIR, 'tier2_t23_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
log(f'saved {summary_path}')

try:
    from google.colab import files
    files.download(CSV_PROBE)
    files.download(summary_path)
except Exception:
    pass